In [13]:
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('../src')
from functions import load_dataset, create_stratified_split, separate_features_target, build_cpg_preprocessing_pipeline

In [14]:
# Load Data
dev_filepath = '../data/development_data.csv'
eval_filepath = '../data/evaluation_data.csv'

dev_df = load_dataset(dev_filepath)
eval_df = load_dataset(eval_filepath)

# Split development data into 80% Train / 20% Validation stratified by age
train_df, val_df = create_stratified_split(
    dev_df, 
    target_col='age', 
    test_size=0.2, 
    random_state=42,
    n_quantiles=10
)

# Verify the Split
print(f"Total Development Data: {dev_df.shape[0]} samples")
print(f"Training Set:           {train_df.shape[0]} samples")
print(f"Validation Set:         {val_df.shape[0]} samples")

Total Development Data: 456 samples
Training Set:           364 samples
Validation Set:         92 samples


In [15]:
# Check the number of missing values in the training set
missing_counts = train_df.isnull().sum()
total_missing = missing_counts.sum()
cols_with_missing = missing_counts[missing_counts > 0]

print(f"Total missing values: {total_missing}")
print(f"Number of columns with missing values: {len(cols_with_missing)} out of {train_df.shape[1]}")

Total missing values: 10992
Number of columns with missing values: 1000 out of 1004


Since the assignment specifies that the missing CpG beta values are Missing Completely at Random (MCAR) and because we have a relatively small sample size which means we cannot simply drop rows, a good idea might be to use median imputation. Below we instantiate a pipeline for feature imputation and scaling.


In [16]:
pipeline = build_cpg_preprocessing_pipeline()

Now we separate our features (CpG sites) from target (Age) and metadata (Sex/Ethnicity). We will treat CpG beta values as our only predictive features (X). We want the model to learn directly from the DNA. If we also include sex and ethnicity as features, the model might use demographics to adjust the age guess rather than relying purely on the biological aging signal.

Instead, we will keep sex and ethnicity in a metadata dataframe to evaluate the model later.

In [17]:
X_train, y_train, meta_train = separate_features_target(train_df)
X_val, y_val, meta_val = separate_features_target(val_df)

In [18]:
# Fit and Transform the TRAINING data
# Calculate the medians and scaling parameters and apply them
X_train_processed = pipeline.fit_transform(X_train)

# Transform the VALIDATION data
# Apply the exact medians and scales learned from the training data to the validation data
X_val_processed = pipeline.transform(X_val)

# Check results
print(f"Original X_train shape: {X_train.shape}")
print(f"Processed X_train shape: {X_train_processed.shape}")

Original X_train shape: (364, 1000)
Processed X_train shape: (364, 1000)
